In [1]:
import requests
from bs4 import BeautifulSoup
from typing import List, Dict, Optional

BASE_URL = "https://www.thehindu.com/latest-news/"

def scrape_the_hindu_page(page: int) -> List[Dict]:
    params = {"page": page}
    resp = requests.get(BASE_URL, params=params, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    items: List[Dict] = []

    # Page-level date (e.g. "Thursday, 8th January 2026")
    page_date_el = soup.select_one("div.latest-news div.latest-date")
    page_date: Optional[str] = (
        page_date_el.get_text(strip=True) if page_date_el else None
    )

    # Each news item on that page
    for li in soup.select("div.latest-news ul.timeline-with-img > li"):
        # headline and url
        title_a = li.select_one("h3.title > a")
        if not title_a:
            continue

        headline = title_a.get_text(strip=True)
        url = title_a.get("href")

        # category (e.g. "Kerala")
        category_a = li.select_one("div.right-content div.label > a")
        category: Optional[str] = (
            category_a.get_text(strip=True) if category_a else None
        )

        # published datetime (ISO) from data-published
        time_div = li.select_one("div.news-time.time")
        published_iso: Optional[str] = None
        if time_div and time_div.has_attr("data-published"):
            published_iso = time_div["data-published"]

        items.append(
            {
                "headline": headline,
                "url": url,
                "category": category,
                "page": page,
                "page_date": page_date,
                "published_datetime": published_iso,
            }
        )

    return items

def scrape_the_hindu_latest_n_pages(n_pages: int = 10) -> List[Dict]:
    all_items: List[Dict] = []
    for page in range(1, n_pages + 1):
        page_items = scrape_the_hindu_page(page)
        all_items.extend(page_items)
    return all_items


In [2]:
import json
from pathlib import Path
import pandas as pd

def save_news_to_files(
    items: List[Dict],
    excel_path: str = "the_hindu_latest.xlsx",
    json_path: str = "the_hindu_latest.json",
) -> None:
    """
    Save scraped news items to an Excel file and a JSON file.

    items: list of dicts returned by scrape_the_hindu_latest_n_pages()
    excel_path: output .xlsx path
    json_path: output .json path
    """

    # Ensure parent dirs exist
    Path(excel_path).parent.mkdir(parents=True, exist_ok=True)
    Path(json_path).parent.mkdir(parents=True, exist_ok=True)

    # Excel
    df = pd.DataFrame(items)
    df.to_csv(excel_path, index=False)

    # JSON (pretty-printed UTF-8)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)

In [3]:
items = scrape_the_hindu_latest_n_pages()
print(f"Items: {items}")

save_news_to_files(
        items,
        excel_path="output/the_hindu_latest_10_pages.csv",
        json_path="output/the_hindu_latest_10_pages.json",
    )


Items: [{'headline': 'BMW Group India bridges gap with MB, clocks sales of 18,001 units in CY25', 'url': 'https://www.thehindu.com/business/bmw-group-india-bridges-gap-with-mb-clocks-sales-of-18001-units-in-cy25/article70487791.ece', 'category': 'Business', 'page': 1, 'page_date': 'Friday, 9th January 2026', 'published_datetime': '2026-01-09T08:24:31+05:30'}, {'headline': 'Supreme Court sets aside A.P. HC order quashing FIRs filed by ACB', 'url': 'https://www.thehindu.com/news/national/andhra-pradesh/supreme-court-sets-aside-ap-hc-order-quashing-firs-filed-by-acb/article70487704.ece', 'category': 'Andhra Pradesh', 'page': 1, 'page_date': 'Friday, 9th January 2026', 'published_datetime': '2026-01-09T08:22:16+05:30'}, {'headline': 'Chief Minister Siddaramaiah urges Kerala Chief Minister Pinarayi Vijayan to withdraw Malayalam Bhasha Bill 2025', 'url': 'https://www.thehindu.com/news/national/karnataka/chief-minister-siddaramaiah-urges-kerala-chief-minister-pinarayi-vijayan-to-withdraw-mala

In [9]:
import json
from pathlib import Path
import pandas as pd

def save_news_to_files(
    items: List[Dict],
    excel_path: str = "the_hindu_latest.xlsx",
    json_path: str = "the_hindu_latest.json",
) -> None:
    """
    Save scraped news items to an Excel file and a JSON file.

    items: list of dicts returned by scrape_the_hindu_latest_n_pages()
    excel_path: output .xlsx path
    json_path: output .json path
    """

    # Ensure parent dirs exist
    Path(excel_path).parent.mkdir(parents=True, exist_ok=True)
    Path(json_path).parent.mkdir(parents=True, exist_ok=True)

    # Excel
    df = pd.DataFrame(items)
    df.to_excel(excel_path, index=False)

    # JSON (pretty-printed UTF-8)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)

In [5]:
df = pd.read_csv('output/Filtered.csv')

UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 15-16: invalid continuation byte